In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from string import punctuation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\heman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
data = pd.read_csv('Language Detection.csv')

In [3]:
data

,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English
...,...,...
10332,ನಿಮ್ಮ ತಪ್ಪು ಏನು ಬಂದಿದೆಯೆಂದರೆ ಆ ದಿನದಿಂದ ನಿಮಗೆ ಒ...,Kannada
10333,ನಾರ್ಸಿಸಾ ತಾನು ಮೊದಲಿಗೆ ಹೆಣಗಾಡುತ್ತಿದ್ದ ಮಾರ್ಗಗಳನ್...,Kannada
10334,ಹೇಗೆ ' ನಾರ್ಸಿಸಿಸಮ್ ಈಗ ಮರಿಯನ್ ಅವರಿಗೆ ಸಂಭವಿಸಿದ ಎ...,Kannada
10335,ಅವಳು ಈಗ ಹೆಚ್ಚು ಚಿನ್ನದ ಬ್ರೆಡ್ ಬಯಸುವುದಿಲ್ಲ ಎಂದು ...,Kannada


In [4]:
def preprocess_text(text):
    """
    Preprocess text data 
    """
    # Convert text to lowercase
    text = text.lower()
    
    # Remove punctuation
    punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
    text = ''.join(char for char in text if char not in punctuation)
    
 
    text = ''.join(char for char in text if not char.isdigit())
    
    words = word_tokenize(text)
    
    
    stemmer = PorterStemmer()
    
    stemmed_words = [stemmer.stem(word) for word in words]
    

    preprocessed_text = ' '.join(stemmed_words)
    
    return preprocessed_text


In [5]:
data['preprocessed_text'] = data['Text'].apply(preprocess_text)
data['preprocessed_text'].head()

0    natur in the broadest sens is the natur physic...
1    natur can refer to the phenomena of the physic...
2    the studi of natur is a larg if not the onli p...
3    although human are part of natur human activ i...
4    the word natur is borrow from the old french n...
Name: preprocessed_text, dtype: object

In [6]:
text = data['preprocessed_text']
language = data['Language']

In [7]:
vectorizer = CountVectorizer()
label_encoder = LabelEncoder()

text_v = vectorizer.fit_transform(text)
language_v = label_encoder.fit_transform(language)

In [8]:
model = MultinomialNB()
model.fit(text_v,language_v)

MultinomialNB()

In [9]:
def language_detector(input_xlanguage):
    
    input_xlanguage_v = vectorizer.transform([input_xlanguage])

    language_detected = (model.predict(input_xlanguage_v)[0])

    return language_detected

In [10]:
def predict(input_text):
    
    predicted_language = label_encoder.inverse_transform((language_detector(input_text)).reshape(-1))
    print(f"The predicted language is: {predicted_language}")

In [11]:
predict("Привет, это тест")
predict("we are boys")
predict("انا مصري ")
predict("mon petit.")

The predicted language is: ['Russian']
The predicted language is: ['English']
The predicted language is: ['Arabic']
The predicted language is: ['French']


In [12]:
model.score(text_v,language_v)

0.9930347296120732

In [13]:
import pickle

file = open('model.pkl','wb')

pickle.dump(model, file)